### Descomposición estacional + Pronóstico 1 año usando SARIMA

### Importación dataset

In [1]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── 1. CARGAR DATOS ──────────────────────────────────────────
df = pd.read_csv('Ingresos Troncal.csv', encoding='latin1')

for col in ['Monto Total Efectivo', 'Monto Total Tarjetas']:
    df[col] = df[col].replace(r'[\$,]', '', regex=True).astype(float)

df['Ingreso Total'] = df['Monto Total Efectivo'] + df['Monto Total Tarjetas']
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Agrupar por semana (toda la serie)
ts_full = df.groupby('Fecha')['Ingreso Total'].sum()
ts_full = ts_full.asfreq('D').interpolate()
ts_full = ts_full.resample('W').sum()

# ── 2. SEPARAR ENTRENAMIENTO (2023-2025) Y REAL 2026 ─────────
ts_train = ts_full[(ts_full.index.year >= 2023) & (ts_full.index.year <= 2025)]
ts_2026  = ts_full[ts_full.index.year == 2026]

print(f"Entrenamiento: {ts_train.index.min().date()} → {ts_train.index.max().date()}")
print(f"Semanas entrenamiento: {len(ts_train)}")
print(f"Semanas reales 2026 disponibles: {len(ts_2026)}")

# ── 3. TEST DE ESTACIONARIEDAD (ADF) ─────────────────────────
print("\n--- Test ADF (estacionariedad) ---")
adf = adfuller(ts_train)
print(f"  Estadístico: {adf[0]:.4f}")
print(f"  p-value:     {adf[1]:.4f}")
if adf[1] < 0.05:
    print("  → Serie ESTACIONARIA (p < 0.05)")
else:
    print("  → Serie NO estacionaria, se aplicará diferenciación (d=1)")

# ── 4. GRÁFICA ACF Y PACF ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts_train.diff().dropna(),  lags=40, ax=axes[0], title='ACF – Serie diferenciada')
plot_pacf(ts_train.diff().dropna(), lags=20, ax=axes[1], title='PACF – Serie diferenciada')
plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n Guardado: acf_pacf.png")

# ── 5. AJUSTAR SARIMA(0,1,1)(0,1,1)[52] ─────────────────────
print("\n--- Ajustando SARIMA(0,1,1)(0,1,1)[52] ---")
print("    (esto puede tardar unos minutos...)")

modelo = SARIMAX(ts_train,
                 order=(0, 1, 1),
                 seasonal_order=(0, 1, 1, 52),
                 enforce_stationarity=False,
                 enforce_invertibility=False)

resultado = modelo.fit(disp=False)
print(resultado.summary())

# ── 6. DIAGNÓSTICOS DEL MODELO ───────────────────────────────
residuos = resultado.resid.dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuos vs tiempo
axes[0, 0].plot(residuos.index, residuos / 1e6, color='steelblue', lw=1)
axes[0, 0].axhline(0, color='red', linestyle='--', lw=1)
axes[0, 0].set_title('Residuos vs Tiempo')
axes[0, 0].set_ylabel('Millones MXN')

# Histograma
axes[0, 1].hist(residuos, bins=20, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Distribución de Residuos')

# ACF de residuos
plot_acf(residuos, lags=20, ax=axes[1, 0], title='ACF – Residuos')

# Q-Q plot
from scipy import stats
stats.probplot(residuos, plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.suptitle('Diagnósticos del Modelo SARIMA(0,1,1)(0,1,1)[52]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('diagnosticos_sarima.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n Guardado: diagnosticos_sarima.png")

# ── 7. PRONÓSTICO 52 SEMANAS (2026) ──────────────────────────
pronostico = resultado.get_forecast(steps=52)
media      = pronostico.predicted_mean
ic         = pronostico.conf_int()

# ── 8. GRÁFICA PRONÓSTICO + REAL 2026 ────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))

# Histórico de entrenamiento
ax.plot(ts_train.index, ts_train / 1e6,
        label='Histórico (2023–2025)', color='steelblue', lw=1.5)

# Pronóstico
ax.plot(media.index, media / 1e6,
        label='Pronóstico SARIMA(0,1,1)(0,1,1)[52]', color='tomato', lw=2)

# Intervalo de confianza 95%
ax.fill_between(ic.index,
                ic.iloc[:, 0] / 1e6,
                ic.iloc[:, 1] / 1e6,
                alpha=0.25, color='orange', label='IC 95%')

# Real 2026 encima del pronóstico
if len(ts_2026) > 0:
    ax.plot(ts_2026.index, ts_2026 / 1e6,
            label='Real 2026', color='seagreen',
            lw=2, marker='o', markersize=4, zorder=5)

ax.axvline(x=ts_train.index[-1], color='gray', linestyle='--',
           lw=1, alpha=0.7, label='Inicio pronóstico')

ax.set_title('Pronóstico de Ingresos 2026 vs Real\nSARIMA(0,1,1)(0,1,1)[52]',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Millones MXN')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pronostico_sarima.png', dpi=150, bbox_inches='tight')
plt.close()
print(" Guardado: pronostico_sarima.png")

# ── 9. RESUMEN NUMÉRICO ──────────────────────────────────────
print("\n--- Pronóstico mensual 2026 ---")
df_fc = pd.DataFrame({'Semana': media.index, 'Pronostico': media.values})
df_fc['Mes'] = df_fc['Semana'].dt.to_period('M')
resumen_fc = df_fc.groupby('Mes')['Pronostico'].sum()

# Real 2026 mensual (si hay datos)
if len(ts_2026) > 0:
    df_real = ts_2026.reset_index()
    df_real.columns = ['Semana', 'Real']
    df_real['Mes'] = df_real['Semana'].dt.to_period('M')
    resumen_real = df_real.groupby('Mes')['Real'].sum()
else:
    resumen_real = pd.Series(dtype=float)

print(f"\n{'Mes':<12} {'Pronóstico':>18} {'Real':>18} {'Diferencia':>18}")
print("-" * 68)
for mes, pron in resumen_fc.items():
    real  = resumen_real.get(mes, np.nan)
    diff  = (real - pron) if not np.isnan(real) else np.nan
    real_str = f"${real:>12,.0f}" if not np.isnan(real) else "   (sin datos)"
    diff_str = f"${diff:>+12,.0f}" if not np.isnan(diff) else ""
    print(f"  {str(mes):<10} ${pron:>12,.0f} MXN   {real_str} MXN   {diff_str}")

print(f"\n  Total pronosticado 2026 : ${resumen_fc.sum():>12,.0f} MXN")
if len(resumen_real) > 0:
    print(f"  Total real 2026 (parcial): ${resumen_real.sum():>12,.0f} MXN")
print(f"\n  AIC del modelo: {resultado.aic:>10.2f}")
print(f"  BIC del modelo: {resultado.bic:>10.2f}")


Entrenamiento: 2023-01-01 → 2025-12-28
Semanas entrenamiento: 157
Semanas reales 2026 disponibles: 8

--- Test ADF (estacionariedad) ---
  Estadístico: -6.3037
  p-value:     0.0000
  → Serie ESTACIONARIA (p < 0.05)

✅ Guardado: acf_pacf.png

--- Ajustando SARIMA(0,1,1)(0,1,1)[52] ---
    (esto puede tardar unos minutos...)
                                     SARIMAX Results                                      
Dep. Variable:                      Ingreso Total   No. Observations:                  157
Model:             SARIMAX(0, 1, 1)x(0, 1, 1, 52)   Log Likelihood                -752.769
Date:                            Wed, 18 Mar 2026   AIC                           1511.538
Time:                                    20:12:53   BIC                           1517.274
Sample:                                01-01-2023   HQIC                          1513.723
                                     - 12-28-2025                                         
Covariance Type:                     